In [1]:
PROJECT_ROOT='/home/featurize/work/biological_prior_validation'

In [2]:
from pathlib import Path
import urllib.request, urllib.error, urllib.parse
import json, re, hashlib, gzip, time, shutil, sys, platform
from datetime import datetime, timezone

ACCESSION='GSM7056149'
SOFT_URL='https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7056149&targ=self&form=text&view=full'

def now():return datetime.now(timezone.utc).isoformat()
def atomic_json(path,obj):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2));tmp.replace(path)

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def geo_files(text):
    if not re.search(r'^\^SAMPLE = GSM7056149\s*$',text,re.M):
        raise ValueError('GEO response is not GSM7056149 SOFT; possible HTML/sign-in/captcha page')
    urls=re.findall(r'^!Sample_supplementary_file_\d+ = (.+)$',text,re.M)
    files=[]
    for u in urls:
        u=u.strip().replace('ftp://','https://',1);parsed=urllib.parse.urlsplit(u)
        if parsed.scheme!='https' or parsed.hostname!='ftp.ncbi.nlm.nih.gov':
            raise ValueError('Unexpected supplementary host')
        name=Path(parsed.path).name
        if not name.startswith(ACCESSION+'_') or not name.endswith('.gz') or name in {x['filename'] for x in files}:
            raise ValueError('Unexpected/duplicate supplementary filename')
        files.append(dict(filename=name,url=u))
    if not files:raise ValueError('No supplementary files found')
    return files

def request(url,headers=None,method=None,timeout=60):
    hdr={'User-Agent':'GSM7056149-reproducible-data-audit/1.0','Accept-Encoding':'identity'}
    hdr.update(headers or {})
    return urllib.request.urlopen(urllib.request.Request(url,headers=hdr,method=method),timeout=timeout)

def probe(url,timeout=60):
    with request(url,method='HEAD',timeout=timeout) as r:
        size=r.headers.get('Content-Length')
        return dict(size_bytes=int(size) if size else None,etag=r.headers.get('ETag'),
                    last_modified=r.headers.get('Last-Modified'),accept_ranges=r.headers.get('Accept-Ranges'))

def gzip_prefix_check(path):
    with Path(path).open('rb') as f:magic=f.read(2)
    if magic!=b'\x1f\x8b':raise ValueError('Not gzip: possible downloaded HTML/error page')
    with gzip.open(path,'rb') as f:prefix=f.read(256)
    if not prefix:raise ValueError('Empty uncompressed prefix')
    return dict(gzip_validation='prefix_only_not_full_CRC',uncompressed_prefix_hex=prefix[:24].hex())

def same_version(old,new):
    for key in ['etag','last_modified','size_bytes']:
        if old.get(key) is not None and new.get(key) is not None and old[key]!=new[key]:return False
    return True

def download_one(item,raw_dir,previous=None,timeout=60,attempts=4,verify_existing=False,progress_every=10):
    raw_dir=Path(raw_dir);raw_dir.mkdir(parents=True,exist_ok=True)
    name=item['filename']
    if Path(name).name!=name:raise ValueError('Unsafe filename')
    dest=raw_dir/name;part=raw_dir/(name+'.part');side=raw_dir/(name+'.part.json')
    remote=item['remote'];total=remote.get('size_bytes')
    if dest.exists():
        if not previous or previous.get('status')!='complete':
            raise FileExistsError('Existing file lacks a completed manifest entry: '+str(dest))
        if previous.get('url')!=item['url'] or not same_version(previous.get('remote',{}),remote):
            raise ValueError('Existing file or GEO remote version changed; preserve it and inspect before replacing')
        if dest.stat().st_size!=previous['size_bytes'] or (total is not None and dest.stat().st_size!=total):
            raise ValueError('Existing file size mismatch')
        if verify_existing and file_hash(dest)!=previous['sha256']:raise ValueError('Existing SHA256 mismatch')
        return dict(previous,existing_check='size_and_remote_validators'+('_and_SHA256' if verify_existing else '_without_rehash'))
    if part.exists():
        if not side.exists():raise ValueError('Partial file has no version sidecar; preserve and inspect: '+str(part))
        saved=json.loads(side.read_text())
        if saved.get('url')!=item['url'] or not same_version(saved['remote'],remote):
            raise ValueError('Partial file remote version changed; do not append incompatible versions')
    atomic_json(side,dict(url=item['url'],remote=remote))
    last_error=None
    for attempt in range(attempts):
        offset=part.stat().st_size if part.exists() else 0
        if total is not None and offset>total:raise ValueError('Partial file exceeds remote size')
        try:
            if not (total is not None and offset==total):
                headers={}
                if offset:
                    headers['Range']=f'bytes={offset}-'
                    etag=remote.get('etag')
                    validator=etag if etag and not etag.startswith('W/') else remote.get('last_modified')
                    if validator:headers['If-Range']=validator
                with request(item['url'],headers,timeout=timeout) as response:
                    status=response.status
                    if status==206:
                        value=response.headers.get('Content-Range','')
                        match=re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)',value)
                        if not match or int(match[1])!=offset:raise ValueError('Invalid resume Content-Range')
                        end=int(match[2]);response_total=int(match[3])
                        if end<offset or response_total<=end:raise ValueError('Invalid range bounds')
                        if total is not None and response_total!=total:raise ValueError('Remote total changed')
                        total=response_total
                        mode='ab' if offset else 'wb'
                    elif status==200:
                        offset=0;mode='wb'
                        length=response.headers.get('Content-Length')
                        if length:
                            if total is not None and int(length)!=total:raise ValueError('Remote size changed')
                            total=int(length)
                    else:raise ValueError('Unexpected HTTP status: '+str(status))
                    h=hashlib.sha256()
                    if offset:
                        with part.open('rb') as f:
                            for b in iter(lambda:f.read(1048576),b''):h.update(b)
                    done=offset;tick=time.monotonic()
                    with part.open(mode) as f:
                        while True:
                            b=response.read(1048576)
                            if not b:break
                            f.write(b);h.update(b);done+=len(b)
                            if time.monotonic()-tick>=progress_every:
                                print(name, f'{done/1048576:.1f} MiB', f'/ {total/1048576:.1f} MiB' if total else '(total unknown)',flush=True)
                                tick=time.monotonic()
                    if total is not None and done!=total:raise IOError(f'Incomplete response: {done}/{total} bytes')
                    sha=h.hexdigest()
            else:sha=file_hash(part)
            validation=gzip_prefix_check(part)
            receipt=dict(filename=name,url=item['url'],remote=remote,status='complete',
                         size_bytes=part.stat().st_size,sha256=sha,completed_utc=now(),
                         checksum_source='locally_computed_not_publisher_reference',**validation)
            atomic_json(raw_dir/(name+'.receipt.json'),receipt)
            part.replace(dest);side.unlink(missing_ok=True)
            print('COMPLETE',name,flush=True)
            return receipt
        except (urllib.error.URLError,TimeoutError,ConnectionError,IOError) as e:
            last_error=e
            if attempt+1<attempts:
                print('Retry',name,str(e),flush=True);time.sleep(min(2**attempt,5))
    raise RuntimeError('Download incomplete; partial retained: '+str(last_error))

def prepare(cfg):
    root=Path(cfg['project_root']);raw=root/'data/raw';audit=root/'data/metadata_audit'
    for sub in ['data/raw','data/metadata_audit','data/splits','data/processed','src','experiments','configs','results/raw','results/summary','figures','reports']:
        (root/sub).mkdir(parents=True,exist_ok=True)
    with request(SOFT_URL,timeout=cfg['timeout']) as r:text=r.read().decode('utf-8')
    (audit/'GSM7056149_geo.soft.txt').write_text(text)
    all_files=geo_files(text)
    selection=cfg.get('filenames')
    if selection is not None:
        unknown=set(selection)-{x['filename'] for x in all_files}
        if unknown:raise ValueError('Selected filenames absent from GEO: '+str(unknown))
        all_files=[x for x in all_files if x['filename'] in selection]
    for item in all_files:
        try:item['remote']=probe(item['url'],cfg['timeout'])
        except Exception as e:item['remote']=dict(size_bytes=None,etag=None,last_modified=None,probe_error=str(e))
    (audit/'download_plan.json').write_text(json.dumps(all_files,indent=2))
    (root/'configs/download_config.json').write_text(json.dumps(cfg,indent=2))
    (root/'results/raw/download_environment.json').write_text(json.dumps(dict(python=sys.version,platform=platform.platform(),accession=ACCESSION,utc=now(),geo_soft_sha256=hashlib.sha256(text.encode()).hexdigest()),indent=2))
    total=sum(x['remote']['size_bytes'] or 0 for x in all_files)
    free=shutil.disk_usage(root).free
    print('Destination:',raw)
    print('Known compressed total:',f'{total/1073741824:.2f} GiB; free {free/1073741824:.2f} GiB')
    print('Unknown-size files:',sum(x['remote']['size_bytes'] is None for x in all_files))
    for item in all_files:print(item['filename'],item['remote']['size_bytes'])
    remaining=0
    for x in all_files:
        p=raw/x['filename'];part=raw/(x['filename']+'.part')
        used=p.stat().st_size if p.exists() else part.stat().st_size if part.exists() else 0
        remaining+=max(0,(x['remote']['size_bytes'] or 0)-used)
    if free<remaining+cfg['reserve_bytes']:raise OSError('Insufficient disk for known download sizes plus reserve')
    return all_files

def download_all(cfg,plan):
    root=Path(cfg['project_root']);raw=root/'data/raw';audit=root/'data/metadata_audit'
    manifest=audit/'download_manifest.json'
    state=json.loads(manifest.read_text()) if manifest.exists() else dict(accession=ACCESSION,files={})
    if state.get('accession')!=ACCESSION:raise ValueError('Manifest accession mismatch')
    for item in plan:
        name=item['filename'];previous=state['files'].get(name)
        receipt=raw/(name+'.receipt.json')
        if (raw/name).exists() and receipt.exists() and (not previous or previous.get('status')!='complete'):
            previous=json.loads(receipt.read_text())
        try:
            result=download_one(item,raw,previous,cfg['timeout'],cfg['attempts'],cfg['verify_existing'])
            state['files'][name]=result
        except Exception as e:
            state['files'][name]=dict(filename=name,url=item['url'],status='failed',error=str(e),utc=now(),remote=item['remote'])
            print('FAILED',name,str(e),flush=True)
        state['updated_utc']=now();atomic_json(manifest,state)
    import pandas as pd
    table=pd.DataFrame([dict(filename=name,status=v['status'],size_bytes=v.get('size_bytes'),sha256=v.get('sha256'),error=v.get('error')) for name,v in state['files'].items()])
    table.to_csv(audit/'download_manifest.csv',index=False)
    failed=[x['filename'] for x in plan if state['files'][x['filename']]['status']!='complete']
    if failed:raise RuntimeError('Some downloads failed; manifest and partial files saved. Re-run this cell after fixing the error: '+str(failed))
    return table


In [3]:
from pathlib import Path
import gzip,json,sys,platform
import pandas as pd
import numpy as np

LINES=['A172','T98G','U87MG']
DRUGS=['Vehicle','Lapatinib','Nintedanib','Trametinib','ZSTK474']

def raw_prefix(path,ng,nc,n=2000):
    with gzip.open(path,'rt') as f:first=f.readline()
    if first.startswith('%%MatrixMarket'):
        raise ValueError('Unexpected real MatrixMarket file; cannot use headerless reader')
    d=pd.read_csv(path,sep=r'\s+',header=None,nrows=n)
    a=d.to_numpy(dtype=float)
    if a.shape[1]!=3 or not np.isfinite(a).all() or (a!=np.floor(a)).any():
        raise ValueError('Expected integer gene_idx/cell_idx/count')
    if (a[:,:2]<1).any() or (a[:,0]>ng).any() or (a[:,1]>nc).any() or (a[:,2]<0).any():
        raise ValueError('Invalid raw coordinate/count')
    return dict(format='headerless_one_based_triplets',scope='prefix_only',rows=len(a),
                first_triplet=a[0].astype(np.int64).tolist(),gene_annotation_rows=ng,
                candidate_barcode_rows=nc)

def count_annotation_rows(path):
    with gzip.open(path,'rt') as f:return sum(1 for line in f)

def initial_raw_audit(root):
    root=Path(root);raw=root/'data/raw';audit=root/'data/metadata_audit'
    state=json.loads((audit/'download_manifest.json').read_text())
    records=[]
    def add(field,value,status,source,limitation=''):
        records.append(dict(field=field,value=str(value),status=status,source_file=str(source),limitation=limitation))
    required=['UMI.count.matrix','cell.annotations.txt','gene.annotations.txt','hash_whitelist_rep1.txt','hash_whitelist_rep2.txt','preprocessed_cds.list.RDS']
    for suffix in required:
        name='GSM7056149_sciPlexGxE_2_'+suffix+'.gz'
        entry=state['files'].get(name,{})
        if entry.get('status')!='complete' or not (raw/name).is_file():
            raise FileNotFoundError('Download not complete for '+name)
    gene_path=raw/'GSM7056149_sciPlexGxE_2_gene.annotations.txt.gz'
    genes=pd.read_csv(gene_path,sep='\t',header=None,names=['gene_id','gene_symbol'],dtype='string',keep_default_na=False)
    genes.insert(0,'gene_idx',range(1,len(genes)+1))
    genes.to_csv(audit/'raw_gene_identifiers.csv',index=False)
    add('raw_gene_annotation_rows',len(genes),'full_annotation_read',gene_path,'Raw annotation universe, not selected or expressed genes')
    cell_path=raw/'GSM7056149_sciPlexGxE_2_cell.annotations.txt.gz'
    nc=count_annotation_rows(cell_path)
    add('raw_cell_annotation_rows',nc,'full_annotation_count',cell_path,'Candidate barcodes, not QC-retained cells; barcode/sample only')
    matrix=raw/'GSM7056149_sciPlexGxE_2_UMI.count.matrix.gz'
    prefix=raw_prefix(matrix,len(genes),nc)
    add('raw_count_format',prefix,'prefix_checked_not_full_scan',matrix,'No MatrixMarket header; no full library size/retained-cell calculation yet')
    designs=[]
    for rep in [1,2]:
        p=raw/f'GSM7056149_sciPlexGxE_2_hash_whitelist_rep{rep}.txt.gz'
        d=pd.read_csv(p,sep='\t',dtype='string',keep_default_na=False,na_values=[''])
        needed={'hash_oligo','hash_plate','cell_line','treatment','dose','replicate'}
        if not needed<=set(d):raise ValueError('Whitelist columns missing: '+str(p))
        if d.hash_oligo.isna().any() or d.hash_oligo.duplicated().any():raise ValueError('Ambiguous hash whitelist key')
        d['source_file']=str(p);d['drug']=d.treatment.str.strip().str.lower().map({x.lower():x for x in DRUGS})
        if d.drug.isna().any() or not d.cell_line.isin(LINES).all():raise ValueError('Unexpected design conditions')
        d['hash_label_well']=d.hash_oligo.str.extract(r'_([A-H][0-9]+)$')[0]
        designs.append(d)
    design=pd.concat(designs,ignore_index=True)
    if design.duplicated(['replicate','hash_oligo']).any():raise ValueError('Duplicate compound hash key')
    design.to_csv(audit/'hash_design_from_raw.csv',index=False)
    grouped=design.groupby(['cell_line','drug','replicate','dose','hash_plate','hash_label_well'],dropna=False).size().rename('n_hash_design_rows').reset_index()
    assert grouped.n_hash_design_rows.sum()==len(design)
    grouped.to_csv(audit/'context_design_counts.csv',index=False)
    contexts=pd.MultiIndex.from_product([LINES,DRUGS],names=['cell_line','drug']).to_frame(index=False)
    contexts['n_cells']=pd.NA;contexts['status']='pending_per_cell_assignment_and_raw_QC'
    # Avoid overwriting previous experiment's context_counts; this belongs to the fresh raw-data audit.
    contexts.to_csv(audit/'context_counts_raw_pending.csv',index=False)
    add('hash_design',len(design),'raw_design_file_read',audit/'hash_design_from_raw.csv','Design rows are not cells or independent replicate units')
    add('biological_independence','UNRESOLVED','experimental_lineage_required','GEO and hash whitelists','Culture/well/date/transduction lineage still needed; neither biological nor technical inferred')
    add('cell_condition_mapping','pending','not_extracted',raw/'GSM7056149_sciPlexGxE_2_preprocessed_cds.list.RDS.gz','RDS is downloaded; extraction/retained-cell mapping not executed here')
    add('retained_cells','pending','not_computed',matrix,'Need complete count/QC or independently verified existing scan before context cell counts')
    add('STRING','not part of this GEO supplementary download','pending_separate_STRING_source','STRING v12 physical','Requires a separate source and gene/protein mapping; cannot substitute GEO files')
    result=pd.DataFrame(records);result.to_csv(audit/'metadata_audit_raw_initial.csv',index=False)
    (root/'results/raw/raw_inventory_environment.json').write_text(json.dumps(dict(python=sys.version,pandas=pd.__version__,platform=platform.platform()),indent=2))
    return result,contexts


In [4]:
raw_audit,pending_contexts=initial_raw_audit(PROJECT_ROOT)
display(raw_audit)
display(pending_contexts)

,field,value,status,source_file,limitation
0,raw_gene_annotation_rows,58347,full_annotation_read,/home/featurize/work/biological_prior_validati...,"Raw annotation universe, not selected or expre..."
1,raw_cell_annotation_rows,43209765,full_annotation_count,/home/featurize/work/biological_prior_validati...,"Candidate barcodes, not QC-retained cells; bar..."
2,raw_count_format,"{'format': 'headerless_one_based_triplets', 's...",prefix_checked_not_full_scan,/home/featurize/work/biological_prior_validati...,No MatrixMarket header; no full library size/r...
3,hash_design,2304,raw_design_file_read,/home/featurize/work/biological_prior_validati...,Design rows are not cells or independent repli...
4,biological_independence,UNRESOLVED,experimental_lineage_required,GEO and hash whitelists,Culture/well/date/transduction lineage still n...
5,cell_condition_mapping,pending,not_extracted,/home/featurize/work/biological_prior_validati...,RDS is downloaded; extraction/retained-cell ma...
6,retained_cells,pending,not_computed,/home/featurize/work/biological_prior_validati...,Need complete count/QC or independently verifi...
7,STRING,not part of this GEO supplementary download,pending_separate_STRING_source,STRING v12 physical,Requires a separate source and gene/protein ma...


,cell_line,drug,n_cells,status
0,A172,Vehicle,<NA>,pending_per_cell_assignment_and_raw_QC
1,A172,Lapatinib,<NA>,pending_per_cell_assignment_and_raw_QC
2,A172,Nintedanib,<NA>,pending_per_cell_assignment_and_raw_QC
3,A172,Trametinib,<NA>,pending_per_cell_assignment_and_raw_QC
4,A172,ZSTK474,<NA>,pending_per_cell_assignment_and_raw_QC
5,T98G,Vehicle,<NA>,pending_per_cell_assignment_and_raw_QC
6,T98G,Lapatinib,<NA>,pending_per_cell_assignment_and_raw_QC
7,T98G,Nintedanib,<NA>,pending_per_cell_assignment_and_raw_QC
8,T98G,Trametinib,<NA>,pending_per_cell_assignment_and_raw_QC
9,T98G,ZSTK474,<NA>,pending_per_cell_assignment_and_raw_QC
